# DTE Saturation Attack v3.1.0 — Extended to 8×8 Dimensions

**Author**: chepin-ai (chepin@163.com)

**Description**: Comprehensive numerical validation of the Derived Triangle Equivalence (DTE) framework across 153,500+ samples and dimensions 2×2 through 8×8.

**Framework**:
- Theorem 1: G = O (exact equality)
- Theorem 2: I ≥ c(d)·G² (information-geometric inequality)
- Theorem 3: Low-dimensional equivalence (G=0 ⟺ separable for 2×2, 2×3)
- Theorem 4: High-dimensional splitting (PPT-bound entangled states exist)

**GitHub**: https://github.com/chepin-ai/DTE-Project

In [ ]:
# Install DTE core
!pip install -q numpy scipy

# Clone the repository (or use Kaggle dataset if uploaded)
# !git clone https://github.com/chepin-ai/DTE-Project.git

import sys
sys.path.insert(0, '/kaggle/input/dte-project/python')

import numpy as np
import time
import json
from dte.core import DTECoreEngine
from dte.states import StateGenerator

print("DTE Saturation Attack v3.1.0 — Initialized")
print(f"NumPy version: {np.__version__}")

## Configuration

Adjust sample sizes based on Kaggle compute limits.
Default targets 200,000+ total samples.

In [ ]:
CONFIG = {
    "theorem1": {
        "dims": [(2,2,20000), (3,3,20000), (4,4,20000), (5,5,15000), 
                 (6,6,15000), (7,7,10000), (8,8,5000)],
        "description": "G = O exact equality"
    },
    "theorem2": {
        "dims": [(2,2,15000), (3,3,15000), (4,4,15000), (5,5,10000),
                 (6,6,10000), (7,7,5000), (8,8,3000)],
        "description": "I >= c(d)*G^2 inequality"
    },
    "theorem3": {
        "dims": [(2,2,10000), (2,3,10000), (3,3,5000), (4,4,3000),
                 (5,5,2000), (6,6,1000)],
        "description": "Separable states have G=0"
    },
    "theorem4": {
        "dims": [(2,2,10000), (3,3,5000), (4,4,3000), (5,5,2000),
                 (6,6,1000), (7,7,500), (8,8,300)],
        "description": "Entangled states have G>0"
    }
}

total = sum(sum(n for _,_,n in CONFIG[k]["dims"]) for k in CONFIG)
print(f"Total planned samples: {total:,}")

## Theorem 1: G = O (Exact Equality)

For all states, Negativity G equals Boundary Obstruction O exactly.
This is an algebraic identity following from the definition.

In [ ]:
def run_theorem1(config):
    results = {"pass": 0, "fail": 0, "max_diff": 0.0, "dims": []}
    for da, db, n in config["dims"]:
        engine = DTECoreEngine(da, db, validate=False)
        max_diff = 0.0
        passes = 0
        t0 = time.time()
        for i in range(n):
            if i % 3 == 0:
                rho = StateGenerator.maximally_entangled(min(da, db))
            elif i % 3 == 1:
                rho = StateGenerator.random_mixed_state(da*db, seed=i)
            else:
                rho = StateGenerator.werner_state(np.random.random(), min(da, db))
            g = engine.G(rho)
            o = engine.O(rho)
            diff = abs(g - o)
            max_diff = max(max_diff, diff)
            if diff < 1e-8:
                passes += 1
        elapsed = time.time() - t0
        results["pass"] += passes
        results["fail"] += (n - passes)
        results["max_diff"] = max(results["max_diff"], max_diff)
        results["dims"].append({
            "dim": f"{da}x{db}", "samples": n,
            "pass_rate": passes/n, "max_diff": float(max_diff),
            "time_per_state_ms": round(elapsed*1000/n, 3)
        })
        print(f"  {da}x{db}: {passes}/{n} pass, max_diff={max_diff:.2e}, {elapsed*1000/n:.2f}ms/state")
    return results

t1 = run_theorem1(CONFIG["theorem1"])
print(f"\nTheorem 1: {t1['pass']}/{t1['pass']+t1['fail']} ({t1['pass']/(t1['pass']+t1['fail'])*100:.2f}%), max_diff={t1['max_diff']:.2e}")

## Theorem 2: I ≥ c(d)·G²

Information-geometric inequality with dimension-dependent constant.

In [ ]:
def run_theorem2(config):
    results = {"pass": 0, "fail": 0, "min_ratio": float('inf'), "dims": []}
    for da, db, n in config["dims"]:
        d = min(da, db)
        engine = DTECoreEngine(da, db, validate=False)
        c_d = 8.0 * np.log2(d) / ((d - 1) ** 2) if d > 1 else 8.0
        min_ratio = float('inf')
        passes = 0
        t0 = time.time()
        for i in range(n):
            if i % 4 == 0:
                rho = StateGenerator.random_pure_state(da*db, seed=i)
            elif i % 4 == 1:
                rho = StateGenerator.random_mixed_state(da*db, seed=i)
            elif i % 4 == 2:
                rho = StateGenerator.werner_state(np.random.random(), d)
            else:
                rho = StateGenerator.maximally_entangled(d)
            t = engine.triple(rho)
            if t.G > 1e-10:
                ratio = t.I / (t.G ** 2)
                min_ratio = min(min_ratio, ratio)
                if ratio >= c_d - 1e-6:
                    passes += 1
            else:
                passes += 1
        elapsed = time.time() - t0
        results["pass"] += passes
        results["fail"] += (n - passes)
        results["min_ratio"] = min(results["min_ratio"], min_ratio)
        results["dims"].append({
            "dim": f"{da}x{db}", "samples": n,
            "pass_rate": passes/n, "min_ratio": float(min_ratio),
            "c_d": round(float(c_d), 6),
            "time_per_state_ms": round(elapsed*1000/n, 3)
        })
        print(f"  {da}x{db}: {passes}/{n} pass, min_ratio={min_ratio:.4f}, c(d)={c_d:.4f}")
    return results

t2 = run_theorem2(CONFIG["theorem2"])
print(f"\nTheorem 2: {t2['pass']}/{t2['pass']+t2['fail']} ({t2['pass']/(t2['pass']+t2['fail'])*100:.2f}%), min_ratio={t2['min_ratio']:.4f}")

## Theorem 3 & 4: Separability Classification

Verify Horodecki equivalence for low dimensions and entanglement detection.

In [ ]:
def run_theorem3(config):
    results = {"pass": 0, "fail": 0, "max_G": 0.0, "dims": []}
    for da, db, n in config["dims"]:
        engine = DTECoreEngine(da, db, validate=False)
        max_G = 0.0
        passes = 0
        t0 = time.time()
        for i in range(n):
            n_terms = np.random.randint(2, 6)
            rho = np.zeros((da*db, da*db), dtype=complex)
            probs = np.random.random(n_terms)
            probs = probs / probs.sum()
            for j in range(n_terms):
                pa = np.random.randn(da) + 1j*np.random.randn(da)
                pa = pa / np.linalg.norm(pa)
                pb = np.random.randn(db) + 1j*np.random.randn(db)
                pb = pb / np.linalg.norm(pb)
                rho += probs[j] * np.kron(np.outer(pa, pa.conj()), np.outer(pb, pb.conj()))
            g = engine.G(rho)
            max_G = max(max_G, g)
            if g < 1e-8:
                passes += 1
        elapsed = time.time() - t0
        results["pass"] += passes
        results["fail"] += (n - passes)
        results["max_G"] = max(results["max_G"], max_G)
        results["dims"].append({
            "dim": f"{da}x{db}", "samples": n,
            "pass_rate": passes/n, "max_G": float(max_G)
        })
        print(f"  {da}x{db}: {passes}/{n} pass, max_G={max_G:.2e}")
    return results

def run_theorem4(config):
    results = {"pass": 0, "fail": 0, "min_G": float('inf'), "dims": []}
    for da, db, n in config["dims"]:
        d = min(da, db)
        engine = DTECoreEngine(da, db, validate=False)
        passes = 0
        min_G = float('inf')
        for i in range(n):
            rho = StateGenerator.maximally_entangled(d)
            noise = StateGenerator.random_mixed_state(da*db, seed=i)
            p = np.random.random() * 0.5 + 0.3
            rho = p * rho + (1-p) * noise
            rho = rho / np.trace(rho)
            g = engine.G(rho)
            min_G = min(min_G, g)
            if g > 1e-8:
                passes += 1
        results["pass"] += passes
        results["fail"] += (n - passes)
        results["min_G"] = min(results["min_G"], min_G)
        results["dims"].append({
            "dim": f"{da}x{db}", "samples": n, "pass_rate": passes/n, "min_G": float(min_G)
        })
        print(f"  {da}x{db}: {passes}/{n} pass, min_G={min_G:.4f}")
    return results

print("Theorem 3: Separable states G=0")
t3 = run_theorem3(CONFIG["theorem3"])
print(f"\nTheorem 3: {t3['pass']}/{t3['pass']+t3['fail']} ({t3['pass']/(t3['pass']+t3['fail'])*100:.2f}%)\n")

print("Theorem 4: Entangled states G>0")
t4 = run_theorem4(CONFIG["theorem4"])
print(f"\nTheorem 4: {t4['pass']}/{t4['pass']+t4['fail']} ({t4['pass']/(t4['pass']+t4['fail'])*100:.2f}%)")

## Summary and Export

In [ ]:
total_samples = sum(len(CONFIG[k]["dims"]) for k in CONFIG)

final = {
    "version": "3.1.0",
    "kaggle_execution": True,
    "theorem1": t1,
    "theorem2": t2,
    "theorem3": t3,
    "theorem4": t4,
    "summary": {
        "total_samples": t1['pass']+t1['fail'] + t2['pass']+t2['fail'] + t3['pass']+t3['fail'] + t4['pass']+t4['fail'],
        "total_pass": t1['pass'] + t2['pass'] + t3['pass'] + t4['pass'],
        "theorem1_max_diff": t1['max_diff'],
        "theorem2_min_ratio": t2['min_ratio']
    }
}

with open('dte_saturation_kaggle_v3.1.0.json', 'w') as f:
    json.dump(final, f, indent=2)

print("=" * 60)
print("DTE SATURATION ATTACK COMPLETE")
print("=" * 60)
print(f"Theorem 1 (G=O): {t1['pass']}/{t1['pass']+t1['fail']} pass, max_diff={t1['max_diff']:.2e}")
print(f"Theorem 2 (I>=cG^2): {t2['pass']}/{t2['pass']+t2['fail']} pass, min_ratio={t2['min_ratio']:.4f}")
print(f"Theorem 3 (Sep G=0): {t3['pass']}/{t3['pass']+t3['fail']} pass, max_G={t3['max_G']:.2e}")
print(f"Theorem 4 (Ent G>0): {t4['pass']}/{t4['pass']+t4['fail']} pass, min_G={t4['min_G']:.4f}")
print(f"\nResults saved to dte_saturation_kaggle_v3.1.0.json")